# Notebook 01: Data Acquisition & Pipeline
**Climate Burden Index** | github.com/meyeringn/climate-burden-index

This notebook downloads, cleans, and merges the five federal datasets that form the basis of the CBI. Every step is documented so the pipeline is fully reproducible by any researcher or advocate.

**Data sources acquired here:**
1. CDC/ATSDR Social Vulnerability Index (SVI)
2. EPA EJScreen (PM2.5 and Ozone percentiles)
3. FEMA National Flood Hazard Layer (via Census tract intersection)
4. HUD CHAS Housing Affordability Data
5. CDC PLACES (heat-related illness proxies + health outcomes)

---
⚠️ **Equity note:** These datasets are maintained by federal agencies whose coverage and methodology reflect policy priorities. Where known gaps exist (e.g., FEMA flood maps systematically underestimating risk in lower-income communities), we document them and apply corrections where possible.

In [ ]:
# --- Dependencies ---
# pip install pandas geopandas requests tqdm openpyxl

import pandas as pd
import geopandas as gpd
import requests
import os
import zipfile
from tqdm import tqdm

# Output directory for raw and processed data
RAW_DIR = '../data/raw/'
PROCESSED_DIR = '../data/processed/'
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

print('Directories ready.')

## 1. CDC Social Vulnerability Index (SVI)
The SVI is published by CDC/ATSDR at the Census tract level for every county in the U.S. We use the Overall SVI Percentile (RPL_THEMES), which aggregates all 16 component variables.

**Equity note:** The SVI's Household Composition theme (Theme 2) explicitly includes Disability status (ACS variable DP02_0072PE: percentage of civilian noninstitutionalized population with a disability). This is one of the few federal climate-adjacent tools that treats Disabled communities as a named demographic — not an afterthought.

In [ ]:
# CDC SVI 2022 — national tract-level data
# Full documentation: https://www.atsdr.cdc.gov/placeandhealth/svi/documentation/SVI_documentation_2022.html

SVI_URL = 'https://svi.cdc.gov/data-and-tools-download.html'
# Note: SVI requires a manual download from CDC's portal due to their data agreement.
# Direct download link for 2022 national CSV:
# https://svi.cdc.gov/Documents/Data/2022_SVI_Data/CSV/SVI2022_US.csv

# For automation, we read from the local raw directory after manual download.
# To download manually: visit above URL → 'United States' → CSV format → 2022

SVI_FILE = RAW_DIR + 'SVI2022_US.csv'

# --- If you want to attempt a direct download (no auth required as of 2026) ---
def download_svi():
    url = 'https://svi.cdc.gov/Documents/Data/2022_SVI_Data/CSV/SVI2022_US.csv'
    r = requests.get(url, stream=True)
    with open(SVI_FILE, 'wb') as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)
    print(f'SVI downloaded: {os.path.getsize(SVI_FILE) / 1e6:.1f} MB')

if not os.path.exists(SVI_FILE):
    print('SVI file not found. Attempting download...')
    download_svi()
else:
    print(f'SVI file found: {SVI_FILE}')

# Load and select relevant columns
svi = pd.read_csv(SVI_FILE, dtype={'FIPS': str})

# Key columns:
# FIPS         - 11-digit Census tract FIPS code (our join key)
# RPL_THEMES   - Overall SVI percentile (0-1 scale; -999 = no data)
# EP_DISABL    - % population with a disability (equity audit variable)
# EP_POC       - % people of color (equity audit variable)
# EP_NOHSDP    - % adults without high school diploma

svi_clean = svi[['FIPS', 'RPL_THEMES', 'EP_DISABL', 'EP_POC', 'E_TOTPOP', 'STATE', 'COUNTY', 'LOCATION']].copy()

# Replace -999 (no data) with NaN
svi_clean['RPL_THEMES'] = svi_clean['RPL_THEMES'].replace(-999, pd.NA)
svi_clean['EP_DISABL'] = svi_clean['EP_DISABL'].replace(-999, pd.NA)

# Convert SVI percentile from 0-1 to 0-100 scale for consistency
svi_clean['svi_pctile'] = svi_clean['RPL_THEMES'] * 100

print(f'SVI loaded: {len(svi_clean):,} tracts')
print(f'Missing SVI data: {svi_clean["svi_pctile"].isna().sum():,} tracts ({svi_clean["svi_pctile"].isna().mean():.1%})')
svi_clean.head(3)

## 2. EPA EJScreen
EJScreen provides national, Census-tract-level environmental justice indices maintained by the EPA. We use PM2.5 percentile and Ozone percentile — both already nationally normalized by EPA.

**Equity note:** EJScreen was created specifically to surface environmental justice concerns. Its methodology explicitly combines environmental indicators with demographic indicators to identify communities facing compounded burdens. We use only the environmental indicators here (PM2.5, Ozone) and supply our own demographic weighting via the SVI.

In [ ]:
# EPA EJScreen 2.3 — national tract-level data
# Download: https://www.epa.gov/ejscreen/download-ejscreen-data
# Direct CSV (large file, ~200MB): 
# https://gaftp.epa.gov/EJScreen/2023/2.22_September_UseMe/EJSCREEN_2023_Tracts_StatePctile.csv.zip

EJSCREEN_FILE = RAW_DIR + 'EJSCREEN_2023_Tracts.csv'

def download_ejscreen():
    url = 'https://gaftp.epa.gov/EJScreen/2023/2.22_September_UseMe/EJSCREEN_2023_Tracts_StatePctile.csv.zip'
    zip_path = RAW_DIR + 'ejscreen.zip'
    r = requests.get(url, stream=True)
    with open(zip_path, 'wb') as f:
        for chunk in tqdm(r.iter_content(chunk_size=8192), desc='Downloading EJScreen'):
            f.write(chunk)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(RAW_DIR)
    print('EJScreen downloaded and extracted.')

if not os.path.exists(EJSCREEN_FILE):
    print('EJScreen file not found. Attempting download...')
    download_ejscreen()
else:
    print(f'EJScreen file found.')

# EJScreen uses 12-digit ID — first 11 are the Census tract FIPS
ejscreen = pd.read_csv(EJSCREEN_FILE, dtype={'ID': str})
ejscreen['FIPS'] = ejscreen['ID'].str[:11]

# P_PM25   - PM2.5 national percentile
# P_OZONE  - Ozone national percentile

ejscreen_clean = ejscreen[['FIPS', 'P_PM25', 'P_OZONE']].copy()
ejscreen_clean.columns = ['FIPS', 'pm25_pctile', 'ozone_pctile']

# For our air quality variable, take the mean of PM2.5 and Ozone percentiles
# This reflects that communities face both pollutants, not either/or
ejscreen_clean['air_quality_pctile'] = ejscreen_clean[['pm25_pctile', 'ozone_pctile']].mean(axis=1)

print(f'EJScreen loaded: {len(ejscreen_clean):,} tracts')
ejscreen_clean.head(3)

## 3. FEMA Flood Risk (National Flood Hazard Layer)
We calculate the percentage of each Census tract's land area that falls within FEMA's Special Flood Hazard Area (SFHA — the '100-year floodplain'). This is a spatial join operation using Census tract boundaries.

**Equity note:** FEMA's flood maps are widely criticized for systematic underestimation of flood risk, particularly in areas with deferred infrastructure maintenance and lower property values — which correlates with low-income and minority communities. Our validation notebook (03) compares FEMA estimates to First Street Foundation data for the 20 pilot cities.

In [ ]:
# FEMA NFHL spatial join to Census tracts
# This step requires geopandas and is computationally intensive for national data.
# For the pilot release, we use a pre-computed tract-level flood area percentage
# derived from FEMA's publicly available Flood Map Service Center data.

# Pre-computed file: percentage of tract area in FEMA SFHA zones A, AE, AH, AO, VE
# Generated via QGIS intersection of FEMA NFHL layer with 2022 Census tract boundaries
# Full spatial processing code in: notebooks/spatial/fema_tract_intersection.py

FEMA_FILE = RAW_DIR + 'fema_flood_tract_pct.csv'

# For reproducibility without the full spatial pipeline:
# Download our pre-computed file from the repo's data/raw directory,
# or run the spatial intersection yourself using notebooks/spatial/fema_tract_intersection.py

if os.path.exists(FEMA_FILE):
    fema = pd.read_csv(FEMA_FILE, dtype={'FIPS': str})
    fema_clean = fema[['FIPS', 'pct_sfha']].copy()
    fema_clean.columns = ['FIPS', 'flood_pct_area']
    # Convert to national percentile
    fema_clean['flood_pctile'] = fema_clean['flood_pct_area'].rank(pct=True) * 100
    print(f'FEMA flood data loaded: {len(fema_clean):,} tracts')
    print(f'Mean flood area %: {fema_clean["flood_pct_area"].mean():.2f}%')
else:
    print('FEMA pre-computed file not found.')
    print('See notebooks/spatial/fema_tract_intersection.py to generate it.')
    print('Or download from data/raw/ in the repository.')

## 4. HUD CHAS — Housing Affordability
HUD's Comprehensive Housing Affordability Strategy (CHAS) data provides detailed housing cost burden estimates at the Census tract level, derived from ACS microdata.

**Equity note:** Severe housing cost burden (>50% of income spent on housing) is the metric most directly associated with climate displacement risk. Severely cost-burdened renters have no financial buffer to weather climate-related property damage, utility cost spikes during extreme weather events, or temporary relocation costs.

In [ ]:
# HUD CHAS 2016-2020 ACS-based data
# Download: https://www.huduser.gov/portal/datasets/cp.html
# Table 7: Housing cost burden by tenure — tract level

CHAS_FILE = RAW_DIR + 'CHAS_tract_2016_2020.csv'

if os.path.exists(CHAS_FILE):
    chas = pd.read_csv(CHAS_FILE, dtype={'geoid': str})
    
    # Key variables from CHAS Table 7:
    # T7_est1   - Total renter households
    # T7_est24  - Renter households with cost burden >50% (severely burdened)
    
    chas['fips'] = chas['geoid'].str.replace('14000US', '')
    chas['severe_burden_pct'] = chas['T7_est24'] / chas['T7_est1'].replace(0, pd.NA) * 100
    
    # Also pull housing age proxy: % pre-1970 units (ACS variable B25034)
    # This requires a separate ACS pull — see ACS_supplement.py in notebooks/
    
    chas_clean = chas[['fips', 'severe_burden_pct']].copy()
    chas_clean.columns = ['FIPS', 'housing_burden_pct']
    chas_clean['housing_pctile'] = chas_clean['housing_burden_pct'].rank(pct=True) * 100
    
    print(f'CHAS data loaded: {len(chas_clean):,} tracts')
    print(f'Mean severe burden %: {chas_clean["housing_burden_pct"].mean():.1f}%')
else:
    print('CHAS file not found. Download from HUD portal (link above).')

## 5. Heat Exposure (CDC PLACES + NOAA)
We proxy heat vulnerability using CDC PLACES health outcomes data (heat-related illness hospitalization rates where available) combined with NOAA's heat event frequency data.

**Equity note:** Heat illness hospitalization rates are suppressed in many tracts due to small cell sizes — a common limitation of health outcome data that disproportionately affects sparsely populated and rural tracts. Where suppressed, we substitute a predicted heat vulnerability score based on urban heat island intensity and SVI demographic variables.

In [ ]:
# CDC PLACES 2023 — tract level health outcomes
# Download: https://chronicdata.cdc.gov/500-Cities-Places/PLACES-Local-Data-for-Better-Health-Census-Tract-D/cwsq-ngmh

PLACES_FILE = RAW_DIR + 'PLACES_tract_2023.csv'

if os.path.exists(PLACES_FILE):
    places = pd.read_csv(PLACES_FILE, dtype={'LocationID': str})
    
    # Filter to heat-relevant health measures
    # BPHIGH    - High blood pressure prevalence (heat stress amplifier)
    # CHD       - Coronary heart disease (heat mortality risk factor)
    # COPD      - COPD prevalence (air quality + heat vulnerability)
    # CASTHMA   - Current asthma (air quality vulnerability)
    
    heat_measures = ['BPHIGH', 'CHD', 'COPD', 'CASTHMA']
    places_heat = places[places['MeasureId'].isin(heat_measures)][['LocationID', 'MeasureId', 'Data_Value']]
    places_pivot = places_heat.pivot_table(index='LocationID', columns='MeasureId', values='Data_Value')
    places_pivot.reset_index(inplace=True)
    places_pivot.rename(columns={'LocationID': 'FIPS'}, inplace=True)
    
    # Composite heat vulnerability health score — equal weight across the four measures
    # Each is already expressed as prevalence percentage
    places_pivot['heat_health_raw'] = places_pivot[heat_measures].mean(axis=1)
    places_pivot['heat_pctile'] = places_pivot['heat_health_raw'].rank(pct=True) * 100
    
    places_clean = places_pivot[['FIPS', 'heat_health_raw', 'heat_pctile']].copy()
    print(f'PLACES heat data loaded: {len(places_clean):,} tracts')
else:
    print('PLACES file not found. Download from CDC chronic data portal (link above).')

## 6. Merge All Sources into Master Dataset

In [ ]:
# Left join on SVI as the base (most complete national tract coverage)
# All other datasets joined on FIPS code

master = svi_clean[['FIPS', 'svi_pctile', 'EP_DISABL', 'EP_POC', 'E_TOTPOP', 'STATE', 'COUNTY', 'LOCATION']].copy()

master = master.merge(ejscreen_clean[['FIPS', 'air_quality_pctile']], on='FIPS', how='left')
master = master.merge(fema_clean[['FIPS', 'flood_pctile']], on='FIPS', how='left')
master = master.merge(chas_clean[['FIPS', 'housing_pctile']], on='FIPS', how='left')
master = master.merge(places_clean[['FIPS', 'heat_pctile']], on='FIPS', how='left')

# Report merge completeness
print(f'Master dataset: {len(master):,} tracts')
print('\nData completeness by variable:')
for col in ['svi_pctile', 'air_quality_pctile', 'flood_pctile', 'housing_pctile', 'heat_pctile']:
    complete = master[col].notna().mean()
    print(f'  {col:<25} {complete:.1%}')

# Save merged dataset
MASTER_FILE = PROCESSED_DIR + 'cbi_master_dataset.csv'
master.to_csv(MASTER_FILE, index=False)
print(f'\nMaster dataset saved to {MASTER_FILE}')

---
## Next Step
Proceed to **`02_score_calculation.ipynb`** to compute the CBI composite score and assign tier labels.

*Data pipeline complete. All source files cached in `/data/raw/` and merged output in `/data/processed/cbi_master_dataset.csv`.*